# H5: checkpoint averaging of the locked RAMP baseline
Uniform weight average of epochs 60/70/80 of `ramp_scratch_l5_s{0,1}`, evaluated with the same fixed-ε AutoAttack protocol (first 1000 test images). About 35 session-minutes on T4 x2. GPU **T4 x2**, Internet **on**, then Save Version → Save & Run All. Result: `ramp_official/ramp_scratch_avg_l5_s{0,1}/eval_autoattack.json` on HF.

In [1]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b claude/read-hand-off-5m8cj2 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

/kaggle/working
/kaggle/working/AAT
=== stage install started 2026-10-04 03:10 UTC ===
/kaggle/working/AAT/baselines/aat_on_ramp.patch:92: trailing whitespace.
                                eps=eps_s, n_iter=args.l_iters[cur_norm_source], is_train=True)               
/usr/local/lib/python3.12/dist-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
OK    import torch            2.10.0+cu128
OK    import torchvision      0.25.0+cu128
OK    import numpy            2.0.2
OK    import yaml             6.0.3
OK    import huggingface_hub  1.11.0
OK    import autoattack       ok
OK    import robustbench      ok
OK    import geotorch         0.4.0
OK    import timm             1.0.26
OK    official RAMP code      RAMP.py, eat_train.py, MAX.py, eva

In [2]:
stage('install', 'bash scripts/ramp_official.sh install')
stage('data', 'bash scripts/ramp_official.sh data')

=== stage install started 2026-10-04 03:11 UTC ===
/usr/local/lib/python3.12/dist-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
OK    import torch            2.10.0+cu128
OK    import torchvision      0.25.0+cu128
OK    import numpy            2.0.2
OK    import yaml             6.0.3
OK    import huggingface_hub  1.11.0
OK    import autoattack       ok
OK    import robustbench      ok
OK    import geotorch         0.4.0
OK    import timm             1.0.26
OK    official RAMP code      RAMP.py, eat_train.py, MAX.py, eval.py import; patches applied; pretr_Linf.pth present
OK    our package             aat imports
OK    GPU                     2 x Tesla T4
OK    Hugging Face            token OK for matokebryan, repo matokebryan/aat-check

In [3]:
stage('avg_eval', 'bash scripts/ramp_official.sh evalavg2 0 1')

=== stage avg_eval started 2026-10-04 03:12 UTC ===
[hub] pulled ramp_official/ramp_scratch_l5_s1/ep_60_0.pth <- matokebryan/aat-checkpoints
[hub] pulled ramp_official/ramp_scratch_l5_s0/ep_60_0.pth <- matokebryan/aat-checkpoints
[hub] pulled ramp_official/ramp_scratch_l5_s1/ep_70_0.pth <- matokebryan/aat-checkpoints
[hub] pulled ramp_official/ramp_scratch_l5_s0/ep_70_0.pth <- matokebryan/aat-checkpoints
[hub] pulled ramp_official/ramp_scratch_l5_s1/ep_80_0.pth <- matokebryan/aat-checkpoints
[hub] pulled ramp_official/ramp_scratch_l5_s0/ep_80_0.pth <- matokebryan/aat-checkpoints
averaged 3 checkpoints -> /kaggle/working/AAT/runs_official/ramp_scratch_avg_l5_s1/avg.pth
averaged 3 checkpoints -> /kaggle/working/AAT/runs_official/ramp_scratch_avg_l5_s0/avg.pth
ramp_scratch_avg: linf eps=0.03137 robust=0.4720
ramp_scratch_avg: linf eps=0.03137 robust=0.4610
ramp_scratch_avg: l2 eps=0.5 robust=0.6570
ramp_scratch_avg: l2 eps=0.5 robust=0.6540
ramp_scratch_avg: l1 eps=12 robust=0.4920
ramp_s